# Notebook 2: Fetch BacDive Physiological Data for PlasticDB Organisms

BacDive (bacdive.dsmz.de) is the world's largest database of standardised microbial strain data.
This notebook scrapes the public BacDive strain pages to retrieve real physiological metadata
for each organism in PlasticDB.

Fields retrieved per organism (where available in BacDive):
- Culture temperature (degrees C) recorded for growth
- pH range recorded for growth
- Oxygen tolerance (aerobic, anaerobic, facultative, etc.)
- Gram stain result
- Cell morphology
- Motility
- Isolation source and geographic location

Results are cached to `../data/bacdive_data.csv`. Re-running skips organisms already in the cache.

Rate: 1.5 requests per second to be polite to the public server.
Organisms not found in BacDive are recorded as `bacdive_found = No`.
No values are estimated or interpolated.

In [ ]:
import re
import time
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

PDB_TSV     = Path.cwd().parent.parent / 'plastic-biodegradation-analysis' / 'data' / 'plasticdb_microorganisms.tsv'
BACDIVE_CSV = Path.cwd().parent / 'data' / 'bacdive_data.csv'
BACDIVE_CSV.parent.mkdir(exist_ok=True)

session = requests.Session()
session.headers['User-Agent'] = 'igem-toronto-research/1.0 (research use, bacdive public pages)'
print('Ready.')

In [ ]:
# Load organism list from PlasticDB
df = pd.read_csv(PDB_TSV, sep='\t', dtype=str, on_bad_lines='skip')
df.columns = [
    'organism','tax_id','plastic','reference','enzyme_name',
    'enzyme_id','db_enzyme_name','gene','genbank_id','sequence',
    'year','evidence','plastic_used','manufacturer','analytical_grade',
    'thermophilic','isolation_sample','isolation_environment',
    'isolation_location','extrapolated_from_enzyme','enzyme_id_in_paper','doi',
]
df['organism'] = df['organism'].str.strip()

orgs = (
    df.groupby('organism')
    .agg(n_entries=('organism','count'))
    .reset_index()
    .sort_values('n_entries', ascending=False)
)
print(f'Organisms: {len(orgs)}')

In [ ]:
def bacdive_search_ids(organism: str, sleep: float = 0.7) -> list:
    """Return BacDive strain IDs for an organism via public search page."""
    time.sleep(sleep)
    url = f'https://bacdive.dsmz.de/?search={requests.utils.quote(organism)}'
    try:
        r = session.get(url, timeout=20)
        r.raise_for_status()
        ids = re.findall(r'href="/strain/(\d+)"', r.text)
        return list(dict.fromkeys(ids))[:3]
    except Exception:
        return []


def bacdive_strain(strain_id: str, sleep: float = 0.7) -> dict:
    """Scrape a BacDive public strain page and return structured fields."""
    time.sleep(sleep)
    url = f'https://bacdive.dsmz.de/strain/{strain_id}'
    try:
        r = session.get(url, timeout=20)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, 'html.parser')
    except Exception:
        return {}

    data = {'bacdive_strain_id': strain_id, 'bacdive_url': url}

    def tbl_values(section_text):
        for h in soup.find_all(['h3','h4']):
            if section_text.lower() in h.get_text().lower():
                tbl = h.find_next('table')
                if tbl:
                    return [td.get_text(strip=True) for td in tbl.find_all('td')]
        return []

    # Temperature
    temps = [v for v in tbl_values('Culture temp') if re.match(r'^\d+\.?\d*$', v)]
    if temps:
        data['bacdive_temp_c'] = '; '.join(temps)

    # pH
    phs = [v for v in tbl_values('pH') if re.match(r'^\d+\.?\d*$', v)]
    if phs:
        data['bacdive_ph'] = '; '.join(phs)

    # Oxygen
    oxy = [v for v in tbl_values('Oxygen') if len(v) > 2 and not v.startswith('@')]
    if oxy:
        data['bacdive_oxygen'] = '; '.join(oxy[:3])

    # Gram stain
    gram = [v for v in tbl_values('Gram') if v and not v.startswith('@')]
    if gram:
        data['bacdive_gram'] = gram[0]

    # Morphology
    morph = [v for v in tbl_values('morphol') if len(v) > 2 and not v.startswith('@')]
    if morph:
        data['bacdive_morphology'] = '; '.join(morph[:4])

    # Motility
    mot = [v for v in tbl_values('Motil') if v and not v.startswith('@')]
    if mot:
        data['bacdive_motility'] = mot[0]

    # Isolation
    isol = [v for v in tbl_values('Isolation') if len(v) > 2 and not v.startswith('@')]
    if isol:
        data['bacdive_isolation'] = '; '.join(isol[:6])

    return data


print('Functions defined.')

In [ ]:
# Test on a well-known organism
test_ids = bacdive_search_ids('Ideonella sakaiensis')
print('Strain IDs found:', test_ids)
if test_ids:
    test_data = bacdive_strain(test_ids[0])
    for k, v in test_data.items():
        print(f'  {k}: {v}')

In [ ]:
# Load cache
if BACDIVE_CSV.exists():
    cached = pd.read_csv(BACDIVE_CSV, dtype=str)
    done   = set(cached['organism'].tolist())
    print(f'Cache: {len(cached)} organisms already fetched.')
else:
    cached = pd.DataFrame()
    done   = set()

todo = orgs[~orgs['organism'].isin(done)].copy()
print(f'Remaining: {len(todo)} organisms to fetch.')

In [ ]:
# Fetch all remaining organisms
rows = []
for i, (_, row) in enumerate(todo.iterrows(), 1):
    org = row['organism']
    ids = bacdive_search_ids(org)
    if not ids:
        rows.append({'organism': org, 'bacdive_found': 'No'})
    else:
        d = bacdive_strain(ids[0])
        d['organism']      = org
        d['bacdive_found'] = 'Yes'
        rows.append(d)
    if i % 25 == 0:
        print(f'  {i}/{len(todo)} done')

if rows:
    new_df   = pd.DataFrame(rows)
    combined = pd.concat([cached, new_df], ignore_index=True) if not cached.empty else new_df
    combined.to_csv(BACDIVE_CSV, index=False)
    print(f'Saved {len(combined)} rows to {BACDIVE_CSV}')
else:
    combined = cached
    print('Nothing new to fetch.')

In [ ]:
# Summary
bd = pd.read_csv(BACDIVE_CSV)
print(f'Total organisms processed: {len(bd)}')
print(f'Found in BacDive:          {(bd["bacdive_found"]=="Yes").sum()}')
print(f'Not found:                 {(bd["bacdive_found"]=="No").sum()}')
print()
print('Organisms found in BacDive with temperature data:')
print(bd[bd['bacdive_found']=='Yes'][['organism','bacdive_temp_c','bacdive_ph','bacdive_gram','bacdive_oxygen']].dropna(subset=['bacdive_temp_c']).head(20).to_string(index=False))